# 問題
General Language Understanding Evaluation (GLUE) ベンチマークで配布されているStanford Sentiment Treebank (SST) をダウンロードし、訓練セット（train.tsv）と開発セット（dev.tsv）のテキストと極性ラベルと読み込み、全てのテキストをトークンID列に変換せよ。このとき、単語埋め込みの語彙でカバーされていない単語は無視し、トークン列に含めないことにせよ。また、テキストの全トークンが単語埋め込みの語彙に含まれておらず、空のトークン列となってしまう事例は、訓練セットおよび開発セットから削除せよ（このため、第7章の実験で得られた正解率と比較できなくなることに注意せよ）。

事例の表現方法は任意でよいが、例えば”contains no wit , only labored gags”がネガティブに分類される事例は、次のような辞書オブジェクトで表現すればよい。

{'text': 'contains no wit , only labored gags',
 'label': tensor([0.]),
 'input_ids': tensor([ 3475,    87, 15888,    90, 27695, 42637])}


この例では、textはテキスト、labelは分類ラベル（ポジティブならtensor([1.])、ネガティブならtensor([0.])）、input_idsはテキストのトークン列をID列で表現している。

In [13]:
# 単語埋め込み語彙の作成
import numpy as np
from gensim.models import KeyedVectors

model = KeyedVectors.load_word2vec_format('./GoogleNews-vectors-negative300.bin', binary=True)
vocab = list(model.key_to_index.keys())
d_emb = model.vector_size
V = len(vocab) + 1

# 埋め込み行列の初期化
E = np.zeros((V, d_emb), dtype=np.float32)

# インデックス対応表
word2id = {'<PAD>': 0}
id2word = {0: '<PAD>'}

# 行列にベクトルを格納
for i, word in enumerate(vocab, start=1):
    E[i] = model[word]
    word2id[word] = i
    id2word[i] = word

In [15]:
word2id["no"]

87

In [34]:
import spacy
from collections import Counter

# --- pathの準備 ---
path_dev = "./SST-2/dev.tsv"
path_train = "./SST-2/train.tsv"

nlp = spacy.load("en_core_web_sm")
def split_sentence(path):
  with open(path, 'r', encoding='utf-8') as f:
    final_list = []
    for row in f:
      row = row.strip().split("\t")
      tmp_dict = {}
      if not row:
          continue
      # カテゴリ宣言行
      if row[1] == "0" or row[1] == "1":
        sentence = row[0]
        word = sentence.strip().split()
        tmp_dict["sentence_word"] = word
        tmp_dict["label"] = int(row[1])
        final_list.append(tmp_dict)
      else:
        print("0と1以外です：", row[1])
    return final_list

In [36]:
split_semtemce_dev = split_sentence(path_dev)
split_semtemce_train = split_sentence(path_train)

0と1以外です： label
0と1以外です： label


In [37]:
split_semtemce_dev

[{'sentence_word': ['it',
   "'s",
   'a',
   'charming',
   'and',
   'often',
   'affecting',
   'journey',
   '.'],
  'label': 1},
 {'sentence_word': ['unflinchingly', 'bleak', 'and', 'desperate'], 'label': 0},
 {'sentence_word': ['allows',
   'us',
   'to',
   'hope',
   'that',
   'nolan',
   'is',
   'poised',
   'to',
   'embark',
   'a',
   'major',
   'career',
   'as',
   'a',
   'commercial',
   'yet',
   'inventive',
   'filmmaker',
   '.'],
  'label': 1},
 {'sentence_word': ['the',
   'acting',
   ',',
   'costumes',
   ',',
   'music',
   ',',
   'cinematography',
   'and',
   'sound',
   'are',
   'all',
   'astounding',
   'given',
   'the',
   'production',
   "'s",
   'austere',
   'locales',
   '.'],
  'label': 1},
 {'sentence_word': ['it',
   "'s",
   'slow',
   '--',
   'very',
   ',',
   'very',
   'slow',
   '.'],
  'label': 0},
 {'sentence_word': ['although',
   'laced',
   'with',
   'humor',
   'and',
   'a',
   'few',
   'fanciful',
   'touches',
   ',',
   '

In [38]:
split_semtemce_dev [0]['sentence_word'][0]

'it'

In [39]:
final_list = []
for i in range(len(split_semtemce_dev)):
  tmp_sentence = split_semtemce_dev [i]['sentence_word'] 
  tmp_dict = {}
  tmp_list = []
  for word in tmp_sentence:
    if word in word2id:
      tmp_list.append(word2id[str(word)])
  tmp_dict["text"] = split_semtemce_dev[i]["sentence_word"]
  tmp_dict["label"] = split_semtemce_dev [i]['label']
  tmp_dict['input_ids'] = tmp_list
  if tmp_dict['input_ids']:
    final_list.append(tmp_dict)
  else:
    print(f'{tmp_dict["text"]}はカラです。')


In [40]:
final_list

[{'text': ['it',
   "'s",
   'a',
   'charming',
   'and',
   'often',
   'affecting',
   'journey',
   '.'],
  'label': 1,
  'input_ids': [16, 13259, 640, 5199, 3900]},
 {'text': ['unflinchingly', 'bleak', 'and', 'desperate'],
  'label': 0,
  'input_ids': [136642, 12607, 4984]},
 {'text': ['allows',
   'us',
   'to',
   'hope',
   'that',
   'nolan',
   'is',
   'poised',
   'to',
   'embark',
   'a',
   'major',
   'career',
   'as',
   'a',
   'commercial',
   'yet',
   'inventive',
   'filmmaker',
   '.'],
  'label': 1,
  'input_ids': [1488,
   165,
   684,
   4,
   953829,
   5,
   6091,
   14671,
   339,
   513,
   15,
   1073,
   507,
   24346,
   11212]},
 {'text': ['the',
   'acting',
   ',',
   'costumes',
   ',',
   'music',
   ',',
   'cinematography',
   'and',
   'sound',
   'are',
   'all',
   'astounding',
   'given',
   'the',
   'production',
   "'s",
   'austere',
   'locales',
   '.'],
  'label': 1,
  'input_ids': [12,
   2527,
   10358,
   637,
   37102,
   1868,
 

In [47]:
import torch
# -------------------------------
# tensor化 (paddingも含める場合)
# -------------------------------
max_len = max(len(item["input_ids"]) for item in final_list)

input_tensors = []
label_tensors = []

for item in final_list:
    ids = item["input_ids"]
    label = item["label"]

    # PAD埋め
    padded = ids + [0] * (max_len - len(ids))
    input_tensors.append(padded)
    label_tensors.append(label)

# PyTorch Tensor に変換
input_tensor = torch.tensor(input_tensors, dtype=torch.long)
label_tensor = torch.tensor(label_tensors, dtype=torch.long)

print(input_tensor.shape)  # (サンプル数, 最大系列長)
print(label_tensor.shape)  # (サンプル数,)

torch.Size([872, 38])
torch.Size([872])


In [42]:
input_tensor 

tensor([[    16,  13259,    640,  ...,      0,      0,      0],
        [136642,  12607,   4984,  ...,      0,      0,      0],
        [  1488,    165,    684,  ...,      0,      0,      0],
        ...,
        [  1256,     88,     29,  ...,      0,      0,      0],
        [     5,     28,  73117,  ...,      0,      0,      0],
        [   380,  54575,  44396,  ...,      0,      0,      0]])

In [43]:
label_tensor

tensor([1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1,
        1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1,
        1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1,
        1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0,
        0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0,
        1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1,
        0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0,
        1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1,
        1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1,
        1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1,
        1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1,
        0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1,
        1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0,

In [ ]:
with open(path_dev, 'r', encoding='utf-8') as f:
    answer_list = []
    for row, label, input in zip(f, label_tensor, input_tensor):
      tmp_ans = {}
      row = row.strip().split("\t")
      tmp_ans["text"] = row[0]
      tmp_ans['label'] = label
      tmp_ans["input_ids"] = input
      answer_list.append(tmp_ans)

In [45]:
answer_list

[{'text': 'sentence',
  'label': tensor(1),
  'input_ids': tensor([   16, 13259,   640,  5199,  3900,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0])},
 {'text': "it 's a charming and often affecting journey . ",
  'label': tensor(0),
  'input_ids': tensor([136642,  12607,   4984,      0,      0,      0,      0,      0,      0,
               0,      0,      0,      0,      0,      0,      0,      0,      0,
               0,      0,      0,      0,      0,      0,      0,      0,      0,
               0,      0,      0,      0,      0,      0,      0,      0,      0,
               0,      0])},
 {'text': 'unflinchingly bleak and desperate ',
  'label': tensor(1),
  'input_ids': tensor([  1488,    165,    684,      4, 953829,      5,   6091,  14671,    339,
             513